# VaiiixBRxAprende no Google Colab

Fluxo gratuito:
1. Baixa do GitHub os resumos diários do serviço principal e do worker de notícias.
2. Monta dataset notícia + preço + decisões.
3. Treina um modelo leve para prever viés intraday (`up` / `down`).
4. Salva artefatos e estatísticas.
5. Sobe os artefatos de volta ao GitHub para a VAIIIxBR consumir.


In [ ]:
import base64, io, json, os, requests, joblib
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from scipy import sparse

# Preencha manualmente no Colab ou use secrets / variáveis de ambiente.
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '')
GITHUB_OWNER = os.getenv('GITHUB_REPO_OWNER', 'SEU_USUARIO')
GITHUB_REPO = os.getenv('GITHUB_REPO_NAME', 'SEU_REPO')
GITHUB_BRANCH = os.getenv('GITHUB_REPO_BRANCH', 'main')
assert GITHUB_TOKEN, 'Defina GITHUB_TOKEN no Colab antes de rodar.'

HEADERS = {
    'Accept': 'application/vnd.github+json',
    'Authorization': f'Bearer {GITHUB_TOKEN}',
    'X-GitHub-Api-Version': '2022-11-28',
}
BASE = f'https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents'


In [ ]:
def gh_list(path):
    r = requests.get(f'{BASE}/{path}', headers=HEADERS, timeout=60)
    r.raise_for_status()
    return r.json()

def gh_get_json(path):
    r = requests.get(f'{BASE}/{path}', headers=HEADERS, timeout=60)
    r.raise_for_status()
    payload = r.json()
    content = base64.b64decode(payload['content']).decode('utf-8')
    return json.loads(content)

def gh_put_bytes(path, data_bytes, message):
    r0 = requests.get(f'{BASE}/{path}', headers=HEADERS, timeout=60)
    sha = r0.json().get('sha') if r0.status_code == 200 else None
    body = {
        'message': message,
        'branch': GITHUB_BRANCH,
        'content': base64.b64encode(data_bytes).decode('utf-8'),
    }
    if sha:
        body['sha'] = sha
    r = requests.put(f'{BASE}/{path}', headers=HEADERS, json=body, timeout=60)
    r.raise_for_status()
    return r.json()


In [ ]:
root_items = gh_list('vaiiixbr/daily')
days = sorted([x['name'] for x in root_items if x['type'] == 'dir'])
print('dias encontrados:', len(days), days[-10:])

rows = []
for day in days:
    try:
        service = gh_get_json(f'vaiiixbr/daily/{day}/service_summary.json')
        news = gh_get_json(f'vaiiixbr/daily/{day}/news_summary.json')
    except Exception as e:
        print('pulando', day, e)
        continue
    ps = service['daily_journal']['price_summary']
    nl = service.get('latest_news_insight', {}) or {}
    for h in news['news_daily'].get('headlines', []):
        title = h.get('title') or ''
        rows.append({
            'date': day,
            'title': title,
            'source': h.get('source'),
            'day_return_pct': ps.get('day_return_pct', 0.0),
            'news_price_score': nl.get('news_price_score', 0.0),
            'headline_count': nl.get('headline_count', 0),
            'price_bias': nl.get('price_bias', 'NEUTRAL'),
            'target': 1 if ps.get('day_return_pct', 0.0) > 0 else 0,
        })

df = pd.DataFrame(rows)
print(df.shape)
df.head()


In [ ]:
assert len(df) >= 20, 'Poucos dados ainda; rode novamente após acumular mais pregões.'
X_text = df['title'].fillna('')
X_num = df[['news_price_score', 'headline_count']].fillna(0.0).astype(float).values
y = df['target'].astype(int).values

vec = TfidfVectorizer(max_features=4000, ngram_range=(1,2), min_df=1)
Xt = vec.fit_transform(X_text)
X = sparse.hstack([Xt, sparse.csr_matrix(X_num)])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
clf = LogisticRegression(max_iter=500, class_weight='balanced')
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:,1]
print(classification_report(y_test, pred, digits=4))
print('samples=', len(df), 'positive_rate=', float(np.mean(y)))


In [ ]:
artifacts_dir = Path('/content/vaiiixbr_artifacts')
artifacts_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(clf, artifacts_dir / 'news_day_model.joblib')
joblib.dump(vec, artifacts_dir / 'news_vectorizer.joblib')
dataset_preview = df.tail(200)
dataset_preview.to_csv(artifacts_dir / 'training_preview.csv', index=False)

stats = {
    'samples': int(len(df)),
    'positive_rate': float(np.mean(y)),
    'updated_at': pd.Timestamp.utcnow().isoformat(),
    'days_used': days,
}
Path(artifacts_dir / 'stats.json').write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding='utf-8')
stats


In [ ]:
for filename in ['news_day_model.joblib', 'news_vectorizer.joblib', 'training_preview.csv', 'stats.json']:
    data = (artifacts_dir / filename).read_bytes()
    gh_put_bytes(f'vaiiixbr/artifacts/{filename}', data, f'Update {filename} from Colab training')
print('artefatos enviados ao GitHub com sucesso')
